# Modelización inicial - Lead Scoring

Este notebook implementa la fase de modelización de `ds-09-modelizar`.
La primera búsqueda se limita a regresión logística para mantener interpretabilidad.
La validación externa (`validation.pkl`) queda reservada para una fase posterior.

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import sklearn
import xgboost
from scipy.stats import loguniform
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "03_notebooks" else Path.cwd()
NOTEBOOK_PATH = PROJECT_ROOT / "03_notebooks" / "07_Modelizacion.ipynb"
COPILOT_PATH = PROJECT_ROOT / ".github" / "copilot-instructions.md"
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost: {xgboost.__version__}")

scikit-learn: 1.9.1
xgboost: 3.4.1


La ruta del dataframe se obtiene de la sección de estado del proyecto,
no se redefine manualmente. También se verifica que el target sea binario
y se muestra su distribución antes de preparar la muestra estratificada.

In [2]:
state_text = COPILOT_PATH.read_text(encoding="utf-8")
match = re.search(r"\*\*Dataframe actual\*\*:\s*`([^`]+)`", state_text)
assert match, "No se encontró Dataframe actual en copilot-instructions.md"
input_path = (COPILOT_PATH.parent / match.group(1)).resolve()
df = pd.read_pickle(input_path)
TARGET = "compra"
assert TARGET in df.columns
assert df[TARGET].nunique() == 2
print(f"Dataframe: {input_path}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("Distribución del target:")
print(df[TARGET].value_counts().rename_axis(TARGET).to_frame("cantidad").assign(proporcion=lambda x: x["cantidad"] / len(df)))

Dataframe: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\03_Entrenamiento\05_train_tablon_preseleccion.pkl
Dimensiones: 6360 filas x 41 columnas
Distribución del target:
        cantidad  proporcion
compra                      
0           3976    0.625157
1           2384    0.374843


Se separan predictoras y target y se toma el 80% del tablón con muestreo estratificado.
La muestra conserva la prevalencia de `compra`; luego se reserva un 20% de esa muestra
para interpretar el modelo sin tocar el fichero externo de validación.

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_sample, _, y_sample, _ = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_search, X_interpret, y_search, y_interpret = train_test_split(
    X_sample, y_sample, test_size=0.20, stratify=y_sample, random_state=RANDOM_STATE
)
print(f"Muestra estratificada: {len(X_sample)} filas")
print(f"Filas para búsqueda CV: {len(X_search)}")
print(f"Filas reservadas para interpretabilidad: {len(X_interpret)}")
print(f"Proporción positiva en muestra: {y_sample.mean():.4f}")
print(f"Validación externa reservada (no utilizada): {PROJECT_ROOT / '02_datos' / '02_Validacion' / 'validation.pkl'}")

Muestra estratificada: 5088 filas
Filas para búsqueda CV: 4070
Filas reservadas para interpretabilidad: 1018
Proporción positiva en muestra: 0.3748
Validación externa reservada (no utilizada): C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\02_Validacion\validation.pkl


El experimento usa cinco folds estratificados y ROC AUC como métrica de selección.
Se exploran 30 configuraciones de `C` y penalización L1/L2 con el solver `saga`.
Recall, precisión, F1 y accuracy se calculan como métricas complementarias.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}
model = LogisticRegression(solver="saga", max_iter=5000, random_state=RANDOM_STATE)
param_distributions = {
    "C": loguniform(1e-3, 1e2),
    "penalty": ["l1", "l2"],
}
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring=scoring,
    refit="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=False,
)
print("Costo estimado: 30 configuraciones x 5 folds = 150 ajustes")
print("CV configurado correctamente:", cv)

Costo estimado: 30 configuraciones x 5 folds = 150 ajustes
CV configurado correctamente: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


Esta celda ejecuta la búsqueda sobre la muestra de entrenamiento de modelización.
No lee ni modifica `02_datos/02_Validacion/validation.pkl`.
El resultado se ordena por ROC AUC medio y conserva las métricas complementarias.

In [5]:
search.fit(X_search, y_search)
results = pd.DataFrame(search.cv_results_)
results["parametros"] = results["params"].map(lambda p: json.dumps(p, sort_keys=True))
ranking = (
    results[[
        "rank_test_roc_auc", "mean_test_roc_auc", "std_test_roc_auc",
        "mean_test_recall", "mean_test_precision", "mean_test_f1",
        "mean_test_accuracy", "parametros"
    ]]
    .sort_values(["rank_test_roc_auc", "std_test_roc_auc"])
    .reset_index(drop=True)
)
print("Ranking de configuraciones por ROC AUC:")
print(ranking.head(10).to_string(index=False))
print("\nMejor configuración:")
print(pd.DataFrame([{
    "algoritmo": "LogisticRegression",
    "roc_auc_medio": search.best_score_,
    "roc_auc_std": results.loc[search.best_index_, "std_test_roc_auc"],
    "recall_medio": results.loc[search.best_index_, "mean_test_recall"],
    "precision_media": results.loc[search.best_index_, "mean_test_precision"],
    "f1_medio": results.loc[search.best_index_, "mean_test_f1"],
    "accuracy_media": results.loc[search.best_index_, "mean_test_accuracy"],
    "parametros": json.dumps(search.best_params_, sort_keys=True),
}]).to_string(index=False))

Ranking de configuraciones por ROC AUC:
 rank_test_roc_auc  mean_test_roc_auc  std_test_roc_auc  mean_test_recall  mean_test_precision  mean_test_f1  mean_test_accuracy                                 parametros
                 1           0.891904          0.013579          0.735082             0.780994      0.757063            0.823096 {"C": 14.528246637516036, "penalty": "l2"}
                 2           0.891862          0.013450          0.734426             0.779686      0.756110            0.822359  {"C": 55.51721685244721, "penalty": "l2"}
                 3           0.891853          0.013455          0.734426             0.779686      0.756110            0.822359  {"C": 73.92266140516048, "penalty": "l1"}
                 4           0.891709          0.013888          0.731803             0.779564      0.754700            0.821622 {"C": 3.4702669886504163, "penalty": "l2"}
                 5           0.891539          0.013635          0.729836             0.777437      

## Interpretabilidad del mejor modelo

Se ajusta la mejor configuración sobre los datos de búsqueda y se analizan sus coeficientes.
Además, se calcula permutation importance sobre el subconjunto reservado para interpretación.
Los coeficientes explican dirección y magnitud; la permutación muestra impacto predictivo observado.

In [6]:
from sklearn.inspection import permutation_importance

best_model = search.best_estimator_
best_model.fit(X_search, y_search)
coef_table = pd.DataFrame({
    "variable": X_search.columns,
    "coeficiente": best_model.coef_[0],
})
coef_table["abs_coeficiente"] = coef_table["coeficiente"].abs()
coef_table = coef_table.sort_values("abs_coeficiente", ascending=False).reset_index(drop=True)
perm = permutation_importance(
    best_model, X_interpret, y_interpret, scoring="roc_auc",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
perm_table = pd.DataFrame({
    "variable": X_interpret.columns,
    "importancia_permutacion": perm.importances_mean,
    "std_permutacion": perm.importances_std,
}).sort_values("importancia_permutacion", ascending=False).reset_index(drop=True)
print("Top 15 coeficientes por valor absoluto:")
print(coef_table.head(15).to_string(index=False))
print("\nTop 15 variables por permutation importance (ROC AUC):")
print(perm_table.head(15).to_string(index=False))

Top 15 coeficientes por valor absoluto:
                        variable  coeficiente  abs_coeficiente
              score_actividad_mm     8.265706         8.265706
      tiempo_en_site_total_yj_mm     4.686541         4.686541
             ocupacion_Housewife     4.130648         4.130648
          ult_actividad_SMS Sent     3.311186         3.311186
                 score_perfil_mm     2.902972         2.902972
  ocupacion_Working Professional     2.218297         2.218297
             ult_actividad_Otros     2.176946         2.176946
           visitas_total_missing    -2.065179         2.065179
      ult_actividad_Email Opened     1.996220         1.996220
     paginas_vistas_visita_yj_mm    -1.897575         1.897575
           fuente_Direct Traffic    -1.638056         1.638056
              ambito_Desconocido    -1.594185         1.594185
ult_actividad_Email Link Clicked     1.592360         1.592360
              origen_Lead Import    -1.496695         1.496695
               

La configuración candidata queda congelada para revisión antes de persistir artefactos.
La selección se basa en el mayor ROC AUC medio de la búsqueda, no en la validación externa.
La evaluación final sobre `validation.pkl` queda explícitamente para la fase posterior.

In [7]:
final_candidate = {
    "algoritmo": "LogisticRegression",
    "parametros": {
        "C": float(search.best_params_["C"]),
        "penalty": search.best_params_["penalty"],
        "solver": "saga",
        "max_iter": 5000,
        "random_state": RANDOM_STATE,
    },
    "metrica_principal": "roc_auc",
    "metricas_cv": {
        "roc_auc_mean": float(search.best_score_),
        "roc_auc_std": float(results.loc[search.best_index_, "std_test_roc_auc"]),
        "recall_mean": float(results.loc[search.best_index_, "mean_test_recall"]),
        "precision_mean": float(results.loc[search.best_index_, "mean_test_precision"]),
        "f1_mean": float(results.loc[search.best_index_, "mean_test_f1"]),
        "accuracy_mean": float(results.loc[search.best_index_, "mean_test_accuracy"]),
    },
}
print(pd.DataFrame([
    {"algoritmo": final_candidate["algoritmo"], "C": final_candidate["parametros"]["C"],
     "penalty": final_candidate["parametros"]["penalty"], "roc_auc_mean": final_candidate["metricas_cv"]["roc_auc_mean"],
     "roc_auc_std": final_candidate["metricas_cv"]["roc_auc_std"]}
]).to_string(index=False))

         algoritmo         C penalty  roc_auc_mean  roc_auc_std
LogisticRegression 14.528247      l2      0.891904     0.013579


## Persistencia de artefactos finales

La configuración congelada se evalúa con predicciones fuera de muestra mediante validación cruzada.
Se generan las curvas ROC, precision-recall y gain/lift sobre la muestra de modelización.
El fichero `validation.pkl` permanece reservado para la evaluación externa posterior.

In [8]:
from pathlib import Path
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

results_dir = PROJECT_ROOT / "06_resultados" / "Modelizacion"
results_dir.mkdir(parents=True, exist_ok=True)
frozen_model = LogisticRegression(C=float(search.best_params_["C"]), penalty=search.best_params_["penalty"], solver="saga", max_iter=5000, random_state=RANDOM_STATE)
cv_prob = cross_val_predict(frozen_model, X_sample, y_sample, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
fpr, tpr, _ = roc_curve(y_sample, cv_prob)
precision_curve, recall_curve, _ = precision_recall_curve(y_sample, cv_prob)
order = np.argsort(-cv_prob)
y_sorted = y_sample.to_numpy()[order]
cum_gain = np.cumsum(y_sorted) / y_sorted.sum()
population = (np.arange(len(y_sorted)) + 1) / len(y_sorted)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(fpr, tpr, label=f"ROC AUC = {auc(fpr, tpr):.3f}"); axes[0].plot([0, 1], [0, 1], "--", color="gray"); axes[0].set(title="Curva ROC", xlabel="FPR", ylabel="TPR"); axes[0].legend()
axes[1].plot(recall_curve, precision_curve); axes[1].set(title="Curva precision-recall", xlabel="Recall", ylabel="Precision")
axes[2].plot(population, cum_gain, label="Modelo"); axes[2].plot([0, 1], [0, 1], "--", color="gray", label="Aleatorio"); axes[2].set(title="Gain acumulado", xlabel="Proporci\u00f3n contactada", ylabel="Compras acumuladas"); axes[2].legend()
fig.tight_layout(); charts_path = results_dir / "curvas_modelo.png"; fig.savefig(charts_path, dpi=150, bbox_inches="tight"); plt.close(fig)
ranking_export = results.copy(); ranking_export["algoritmo"] = "LogisticRegression"; ranking_export["parametros"] = ranking_export["params"].map(lambda p: json.dumps(p, sort_keys=True))
ranking_export = ranking_export[["algoritmo", "rank_test_roc_auc", "mean_test_roc_auc", "std_test_roc_auc", "mean_test_recall", "mean_test_precision", "mean_test_f1", "mean_test_accuracy", "parametros"]].sort_values(["rank_test_roc_auc", "std_test_roc_auc"])
ranking_export.to_csv(results_dir / "ranking_modelos.csv", index=False); ranking_export.to_json(results_dir / "ranking_modelos.json", orient="records", indent=2)
config = {"tipo_proyecto": "clasificacion_binaria", "dataset": {"ruta": str(input_path), "muestra_filas": int(len(X_sample)), "filas_busqueda_cv": int(len(X_search)), "proporcion_positiva": float(y_sample.mean()), "validacion_externa_reservada": str(PROJECT_ROOT / "02_datos" / "02_Validacion" / "validation.pkl")}, "algoritmo": "LogisticRegression", "parametros": final_candidate["parametros"], "cv": {"tipo": "StratifiedKFold", "folds": 5, "random_state": RANDOM_STATE, "metrica_principal": "roc_auc"}, "metricas_cv": final_candidate["metricas_cv"], "interpretabilidad": {"coeficientes": coef_table.head(15).to_dict(orient="records"), "permutation_importance": perm_table.head(15).to_dict(orient="records")}, "graficos": str(charts_path), "nota": "M\u00e9tricas calculadas por validaci\u00f3n cruzada sobre la muestra de entrenamiento; validation.pkl se evaluar\u00e1 en una fase posterior."}
(results_dir / "config_mejor_modelo.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
r = ranking_export.head(10).to_string(index=False)
report = f"""# Informe de modelizaci\u00f3n - Lead Scoring

## Objetivo y problema

El objetivo es priorizar leads con mayor probabilidad de `compra`, manteniendo interpretabilidad para el equipo comercial. El problema es de clasificaci\u00f3n binaria.

## Datos y validaci\u00f3n

- Dataset: `{input_path}`
- Filas totales: {len(df)}
- Muestra estratificada: {len(X_sample)} filas
- Positivos (`compra=1`): {y_sample.mean():.2%}
- B\u00fasqueda: {len(X_search)} filas
- Interpretabilidad: {len(X_interpret)} filas reservadas
- Validaci\u00f3n externa reservada: `02_datos/02_Validacion/validation.pkl`

## Algoritmo y experimento

Se evalu\u00f3 `LogisticRegression` con `RandomizedSearchCV`, 30 configuraciones, `StratifiedKFold` de 5 folds y `roc_auc` como m\u00e9trica principal. Se buscaron `C` y penalizaciones L1/L2 con `solver=\"saga\"`, sin ponderaci\u00f3n de clases.

## Ranking (top 10)

```text
{r}
```

## Configuraci\u00f3n ganadora

- Algoritmo: `LogisticRegression`
- Par\u00e1metros: `{json.dumps(final_candidate["parametros"], sort_keys=True)}`
- ROC AUC medio: {final_candidate["metricas_cv"]["roc_auc_mean"]:.4f}
- Desviaci\u00f3n est\u00e1ndar ROC AUC: {final_candidate["metricas_cv"]["roc_auc_std"]:.4f}
- Recall medio: {final_candidate["metricas_cv"]["recall_mean"]:.4f}
- Precisi\u00f3n media: {final_candidate["metricas_cv"]["precision_mean"]:.4f}
- F1 medio: {final_candidate["metricas_cv"]["f1_mean"]:.4f}
- Accuracy media: {final_candidate["metricas_cv"]["accuracy_mean"]:.4f}

## Interpretabilidad

Las variables con mayor valor absoluto de coeficiente fueron `score_actividad_mm`, `tiempo_en_site_total_yj_mm` y `ocupacion_Housewife`. Por permutation importance destacaron `ult_actividad_SMS Sent`, `tiempo_en_site_total_yj_mm` y `ult_actividad_Email Opened`. Un coeficiente positivo aumenta el log-odds estimado de compra; uno negativo lo reduce.

Gr\u00e1ficos: `06_resultados/Modelizacion/curvas_modelo.png`.

## Cierre

Las m\u00e9tricas son de validaci\u00f3n cruzada sobre la muestra de entrenamiento. Un agente posterior entrenar\u00e1 el modelo de producci\u00f3n con esta configuraci\u00f3n sobre los datos completos y evaluar\u00e1 sobre `validation.pkl`. La familia de \u00e1rboles queda pendiente.
"""
(results_dir / "informe_modelizacion.md").write_text(report, encoding="utf-8")
line = "**Modelo candidato actual**: `../06_resultados/Modelizacion/config_mejor_modelo.json`"
state = COPILOT_PATH.read_text(encoding="utf-8")
if line not in state: state = state.rstrip() + "\n\n" + line + "\n"
COPILOT_PATH.write_text(state, encoding="utf-8")
print(f"Artefactos generados en {results_dir}")
print(f"ROC AUC CV: {final_candidate['metricas_cv']['roc_auc_mean']:.4f}")
print(f"Gr\u00e1ficos: {charts_path}")

Artefactos generados en C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\06_resultados\Modelizacion
ROC AUC CV: 0.8919
Gráficos: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\06_resultados\Modelizacion\curvas_modelo.png
